In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd
import os 
import duckdb
import time


# Configuration constants
MIN_POSTS_PER_USER = 2

# User of Interests
The user of interest are the ones who posted at least twice in "chunk_0_posts" and have a joining date.

Since we're working on a small dataset "chunk_0_posts" we stick to Pandas. 

In [2]:
# Load the posts data
posts_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)

posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

user_post_counts = posts_df.groupby('did_id').size().reset_index(name='post_count')

# Filter out users with less than MIN_POSTS_PER_USER posts.
active_posters = user_post_counts[user_post_counts['post_count'] >= MIN_POSTS_PER_USER]['did_id'].values
filtered_posts_df = posts_df[posts_df['did_id'].isin(active_posters)]

print(f"\nPosts after filtering users with <{MIN_POSTS_PER_USER} posts: {len(filtered_posts_df)}")
print(f"Active posters (≥{MIN_POSTS_PER_USER} posts): {len(active_posters)}")

Initial posts count: 4994663
Unique users: 130111

Posts after filtering users with <2 posts: 4957527
Active posters (≥2 posts): 92975

Posts after filtering users with <2 posts: 4957527
Active posters (≥2 posts): 92975


In [3]:
# Load filter profiles for users of interest only
profiles_path = "../data/posting/cleaned/profiles.parquet"
profiles_table = pq.read_table(profiles_path)

# Filter to active_posters only
profiles_table = profiles_table.filter(pc.is_in(profiles_table['did_id'], pa.array(active_posters)))
user_of_interests = profiles_table.to_pandas()

print(f"Users of interests (active posters -- ≥{MIN_POSTS_PER_USER} posts -- with a join date): {len(user_of_interests)}")

Users of interests (active posters -- ≥2 posts -- with a join date): 65380


In [4]:
# Create a fast lookup dictionary: did_id -> join_date
join_date_dict = dict(zip(
    user_of_interests['did_id'],
    user_of_interests['created_at']
))

print(f"Join date lookup dictionary created: {len(join_date_dict)} entries")

Join date lookup dictionary created: 65376 entries


In [28]:
# Also save as parquet for portability (optional)
join_date_df = pd.DataFrame({
    'did_id': list(join_date_dict.keys()),
    'created_at': list(join_date_dict.values())
})
join_date_parquet_path = "../data/posting/processed/join_dates.parquet"
join_date_df.to_parquet(join_date_parquet_path, index=False)
print(f"✅ join_date_dict also saved as parquet to: {join_date_parquet_path}")

✅ join_date_dict also saved as parquet to: ../data/posting/processed/join_dates.parquet


# Event Filtering
The criteria for filtering are:
- Events must involve at least one user of interest (active posters with ≥2 posts)
- Events must occur within the specified time window after the user's join date

Here we use duckDB since it shines when you have:
+ Large datasets (millions+ rows)
+ Complex joins and aggregations
+ Need to avoid loading everything into memory

In [5]:
# Generic DuckDB filtering function

def filter_events_duckdb(input_path, output_path, db_name, query):
    """
    Filter events using DuckDB with a custom SQL query.
    
    Args:
        input_path: Path to input parquet file
        output_path: Path to output parquet file
        db_name: Name of the database being filtered (for logging)
        query: SQL query to execute. The parquet table is available as 'events_table'
               and the join dates are available as 'join_date_table'
    
    Returns:
        Dictionary with filtering statistics
    """
    print(f"Filtering {db_name} using DuckDB...")
    start_time = time.time()
    
    conn = duckdb.connect()
    
    # Register join_date_dict as a DuckDB table
    join_date_df = pd.DataFrame({
        'user_id': list(join_date_dict.keys()),
        'join_date': list(join_date_dict.values())
    })
    join_date_table = conn.from_df(join_date_df)
    
    # Read input parquet
    events_table = conn.read_parquet(input_path)
    
    # Get total row count before filtering
    total_rows = conn.execute("SELECT COUNT(*) FROM events_table").fetchall()[0][0]
    
    # Execute the provided query
    filtered_events = conn.execute(query).fetch_arrow_table()
    filtered_rows = filtered_events.num_rows
    
    # Write to parquet
    pq.write_table(filtered_events, output_path, compression='zstd')
    
    elapsed_time = time.time() - start_time
    input_size = os.path.getsize(input_path)
    output_size = os.path.getsize(output_path)
    retention_rate = (filtered_rows / total_rows * 100) if total_rows > 0 else 0
    
    print(f"\n✅ DuckDB filtering completed in {elapsed_time:.2f} seconds")
    print(f"Rows: {total_rows:,} → {filtered_rows:,} ({retention_rate:.1f}% kept)")
    print(f"File size: {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Output: {output_path}")
    
    conn.close()
    
    return {
        'elapsed_time': elapsed_time,
        'total_rows': total_rows,
        'filtered_rows': filtered_rows,
        'retention_rate': retention_rate,
        'input_size_gb': input_size / (1024**3),
        'output_size_gb': output_size / (1024**3)
    }

In [6]:
# Filter chunk_0_posts with 14-day window (single-user events)
posts_input_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_output_path = "../data/posting/filtered/chunk_0_posts.parquet"

posts_query = """
SELECT e.*
FROM events_table e
LEFT JOIN join_date_table jd ON e.did_id = jd.user_id
WHERE jd.user_id IS NOT NULL 
  AND EXTRACT(DAY FROM (e.created_at - jd.join_date)) BETWEEN 0 AND 14
"""

stats_posts = filter_events_duckdb(posts_input_path, posts_output_path, "posts", posts_query)
print(f"Posts filtered: {stats_posts['filtered_rows']:,} posts from {stats_posts['total_rows']:,} total")

Filtering posts using DuckDB...

✅ DuckDB filtering completed in 14.07 seconds
Rows: 4,994,663 → 492,705 (9.9% kept)
File size: 0.03 GB → 0.00 GB
Output: ../data/posting/filtered/chunk_0_posts.parquet
Posts filtered: 492,705 posts from 4,994,663 total

✅ DuckDB filtering completed in 14.07 seconds
Rows: 4,994,663 → 492,705 (9.9% kept)
File size: 0.03 GB → 0.00 GB
Output: ../data/posting/filtered/chunk_0_posts.parquet
Posts filtered: 492,705 posts from 4,994,663 total


In [7]:
# Filter blocks with 7-day window (two-user events)
blocks_input_path = "../data/posting/cleaned/blocks.parquet"
blocks_output_path = "../data/posting/filtered/blocks.parquet"

blocks_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN join_date_table jd_did ON e.did_id = jd_did.user_id
LEFT JOIN join_date_table jd_subject ON e.subject_id = jd_subject.user_id
WHERE 
    -- Keep if did_id is in users AND within 7 days of joining
    ((jd_did.user_id IS NOT NULL 
      AND EXTRACT(DAY FROM (e.created_at - jd_did.join_date)) BETWEEN 0 AND 7)
    OR
    -- Keep if subject_id is in users AND within 7 days of joining
    (jd_subject.user_id IS NOT NULL 
     AND EXTRACT(DAY FROM (e.created_at - jd_subject.join_date)) BETWEEN 0 AND 7))
"""

stats_blocks = filter_events_duckdb(blocks_input_path, blocks_output_path, "blocks", blocks_query)
print(f"Blocks filtered: {stats_blocks['filtered_rows']:,} blocks from {stats_blocks['total_rows']:,} total")

Filtering blocks using DuckDB...

✅ DuckDB filtering completed in 6.63 seconds
Rows: 120,084,926 → 267,590 (0.2% kept)
File size: 1.45 GB → 0.00 GB
Output: ../data/posting/filtered/blocks.parquet
Blocks filtered: 267,590 blocks from 120,084,926 total

✅ DuckDB filtering completed in 6.63 seconds
Rows: 120,084,926 → 267,590 (0.2% kept)
File size: 1.45 GB → 0.00 GB
Output: ../data/posting/filtered/blocks.parquet
Blocks filtered: 267,590 blocks from 120,084,926 total


In [ ]:
# Filter follows with 7-day window (two-user events)
follows_input_path = "../data/posting/cleaned/follows.parquet"
follows_output_path = "../data/posting/filtered/follows.parquet"

follows_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN join_date_table jd_did ON e.did_id = jd_did.user_id
LEFT JOIN join_date_table jd_subject ON e.subject_id = jd_subject.user_id
WHERE 
    -- Keep if did_id is in users AND within 7 days of joining
    ((jd_did.user_id IS NOT NULL 
      AND EXTRACT(DAY FROM (e.created_at - jd_did.join_date)) BETWEEN 0 AND 7)
    OR
    -- Keep if subject_id is in users AND within 7 days of joining
    (jd_subject.user_id IS NOT NULL 
     AND EXTRACT(DAY FROM (e.created_at - jd_subject.join_date)) BETWEEN 0 AND 7))
"""

stats_follows = filter_events_duckdb(follows_input_path, follows_output_path, "follows", follows_query)
print(f"Follows filtered: {stats_follows['filtered_rows']:,} follows from {stats_follows['total_rows']:,} total")

In [ ]:
# Filter likes with 7-day window (two-user events)
likes_input_path = "../data/posting/cleaned/likes.parquet"
likes_output_path = "../data/posting/filtered/likes.parquet"

likes_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN join_date_table jd_did ON e.did_id = jd_did.user_id
LEFT JOIN join_date_table jd_subject ON e.subject_id = jd_subject.user_id
WHERE 
    -- Keep if did_id is in users AND within 7 days of joining
    ((jd_did.user_id IS NOT NULL 
      AND EXTRACT(DAY FROM (e.created_at - jd_did.join_date)) BETWEEN 0 AND 7)
    OR
    -- Keep if subject_id is in users AND within 7 days of joining
    (jd_subject.user_id IS NOT NULL 
     AND EXTRACT(DAY FROM (e.created_at - jd_subject.join_date)) BETWEEN 0 AND 7))
"""

stats_likes = filter_events_duckdb(likes_input_path, likes_output_path, "likes", likes_query)
print(f"Likes filtered: {stats_likes['filtered_rows']:,} likes from {stats_likes['total_rows']:,} total")